## Tema 4 - Listado 1 - Ejercicio 1

Ejercicio para entrenar un modelo simple de red neuronal utilizando la biblioteca Keras de Tensorflow.

1) En  este  primer  ejercicio  no  se  va  a  realizar  ningún  tipo  de  preprocesamiento  del 
texto. Las etiquetas (labels) serán 1 para la clase positiva y 0 para la clase negativa. 
Se define una longitud para los textos = 10. Se define un tamaño de vocabulario = 50. 
Hay que ir siguiendo los pasos que se indican a continuación: 
2) Construir  el  vocabulario  (los  tokens  diferentes  de  los  textos  de  entrada).  Mostrar 
los tokens del vocabulario con su índice.  

3) Construir la matriz de documentos-términos con el tamaño máximo elegido, donde 
el peso de cada término es el índice en el vocabulario (si no está se le pone valor 0). 

4) Una vez preprocesado el texto, hay que definir el modelo.  Se va a utilizar la clase 
Sequential, que permite añadir diferentes capas de forma secuencial.  

5) La  primera  capa  del  modelo  es  la  capa  de  embeddings,  que  se  encargará  de 
representar  cada  texto  como  una  matriz  de  vectores.  La  matriz  tendrá  como 
número  de  filas  el  valor  de  la  variable  max_length,  es  decir,  la  longitud  de  las 
secuencias  de  entrada.  Como  número  de  columnas,  tendremos  que  decidir  que 
dimensión  queremos  utilizar  para  representar  los  vectores  de  los  tokens.  En  este 
ejemplo,  vamos  a  utilizar  un  tamaño  de  vector  pequeño  (vector_size  =  8), 
porque estamos con un ejemplo de juguete (las dimensiones que se suelen utilizar 
más son 100, 200 o 300). 

La capa Embedding lo que hará será inicializar una matriz por cada texto. Como se 
ha dicho antes la matriz, tendrá una dimensión de max_length x vector_size. 
En este ejercicio, la matriz se inicializa con pesos aleatorios, pero se podría inicializar 
la matriz a partir de un modelo pre-entrenado de word embeddings (lo veremos en 
otro ejercicio). 

Hay que añadir una capa densa para la salida. En este caso, al ser una salida binaria 
como función de activación podemos utilizar sigmoid. Devolverá una probabilidad, 
que si es cercana a 1 entonces la capa de salida devolverá 1, 0 en otro caso.

## Carga de datos

El conjunto de datos son frases sencillas anotadas manualmente con su sentido positivo o negativo. 

* "Estoy un poco harto del día a día, nada mejora" -> Negativo
* "Hoy es un buen día" -> Positivo
* "No se te ve satisfecho con el trabajo" -> Negativo
* "Este paisaje es hermoso y bonito" -> Positivo


In [7]:
sentences = ['Estoy un poco harto del día a día , nada mejora',
             'Hoy es un buen día',
             'No se te ve satisfecho con el trabajo',
             'Este paisaje es hermoso y bonito']

# 1: positivo, 0: negativo
labels = [0,1,0,1]


Hay que preparar los datos de entrenamiento:

* Longitud de las secuencias de texto = 10
* Tamaño del vocabulario = 50
  

In [1]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


2025-06-30 12:52:14.135290: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-30 12:52:14.139638: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-30 12:52:14.177255: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-30 12:52:14.231959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751280734.277845   16507 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751280734.29

In [ ]:
import spacy
import spacy.cli
from tensorflow.keras.preprocessing.sequence import pad_sequences

#Preparar vocabulario
nlp = spacy.load('es_core_news_sm')

def prepare_vocabulary(corpus, vocab_size):
    token_to_idx={}
    id=1 
    for f in corpus:
        doc= nlp(f)
        for token in doc:
            if token.text not in token_to_idx and id <= vocab_size:
                token_to_idx[token.text]= id
                id+=1
    
    return token_to_idx

#Prepara datos de entrenamiento -> codifica las palabras según id (0 si no está y aplica padding)

def prepare_sentences(corpus, vocab, max_length):
    encoded_sentences=[]
    for f in corpus:
        encoded_sentence=[]
        doc= nlp(f)
        for token in doc:
            if token.text in vocab:
                id= vocab[token.text]
            else:
                id=0
            encoded_sentence.append(id)
        encoded_sentences.append(encoded_sentence)

    prepared_sentences = pad_sequences(encoded_sentences, maxlen=max_length, padding='post', truncating='post')
    print("Oraciones originales(",len(corpus),"):")
    print(corpus)  
    print("Oraciones procesadas(",len(prepared_sentences),"):")
    print(prepared_sentences)    

    return prepared_sentences

max_voc_size=50
max_length=10
voc= prepare_vocabulary(sentences, vocab_size=max_voc_size)
encoding_sentenes= prepare_sentences(sentences, voc, max_length)
voc_size= len(voc)+1
print(voc_size)


Oraciones originales( 4 ):
['Estoy un poco harto del día a día , nada mejora', 'Hoy es un buen día', 'No se te ve satisfecho con el trabajo', 'Este paisaje es hermoso y bonito']
Oraciones procesadas( 4 ):
[[ 1  2  3  4  5  6  7  6  8  9]
 [11 12  2 13  6  0  0  0  0  0]
 [14 15 16 17 18 19 20 21  0  0]
 [22 23 12 24 25 26  0  0  0  0]]
27


## CNN

In [56]:
import tensorflow as tf
tf.__version__

'2.19.0'

## Configuramos el modelo

In [61]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Embedding, Conv1D, MaxPool1D

model= Sequential()

#La capa de embedding hace que cada id del texto codificado sea un vector
vector_size=8
model.add(Embedding(max_voc_size, vector_size)) # devolverá matriz vector_size x max_lenght
# Añadir una capa de aplanado (Flatten) para aplanar la entrada, convirtiendo los datos multidimensionales en un vector unidimensional

model.add(Flatten())

# Añadir una capa densa con 1 neurona para una salida binaria con una función de activación sigmoid para clasificación binaria. Devuelve una probabilidad.
# Si la probabilidad es cercana a 1, la capa devuelve 1, y 0 en otro caso. 

model.add(Dense(1, activation='sigmoid'))


print("Red diseñada correctamente")

Red diseñada correctamente


## Compilamos del modelo

In [62]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.build(input_shape=(None, max_length)) 
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 10, 8)          │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481 (1.88 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamos el modelo

In [63]:
from sklearn.model_selection import train_test_split
import numpy as np

# Convertir a NumPy arrays para asegurar compatibilidad y rendimiento
prepared_sentences= np.array(encoding_sentenes)
labels=np.array(labels)

X_train, X_test, y_train, y_test= train_test_split(encoding_sentenes, labels,
    test_size=0.2, random_state=43)

batch_size= 32
epochs=5

history= model.fit(X_train, y_train, validation_data=(X_test, y_test),
 batch_size=batch_size, epochs=epochs)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 611ms/step - accuracy: 0.6667 - loss: 0.6943 - val_accuracy: 0.0000e+00 - val_loss: 0.6932
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6667 - loss: 0.6882 - val_accuracy: 0.0000e+00 - val_loss: 0.6951
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.6667 - loss: 0.6822 - val_accuracy: 0.0000e+00 - val_loss: 0.6971
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.6667 - loss: 0.6762 - val_accuracy: 0.0000e+00 - val_loss: 0.6990
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6667 - loss: 0.6702 - val_accuracy: 0.0000e+00 - val_loss: 0.7010


## Evaluar el modelo

Vamos a evaluarlo sobre el conjunto test. En primer lugar, vamos a obtener las métricas loss y accuracy en dicho conjunto (que no ha sido utilizado en ninguna fase del entrenamiento).


In [64]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Conjunto más amplio de frases de prueba
test_sentences = [
    "No fui al estreno de la película porque nadie me quería acompañar",
    "Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio",
    "Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor",
    "Al final decidí no ir al cine porque estaba cansada",
    "Todo es maravilloso y formidable, muy bonito"
]

# Preparar los datos

prepared_test = prepare_sentences(test_sentences, voc, max_length)

# Realizar predicciones
predictions = model.predict(prepared_test)

# Interpretar las predicciones con más detalle
print("Predicciones detalladas:")
for i, sentence in enumerate(test_sentences):
    pred = predictions[i][0]
    sentiment = "Positivo" if pred > 0.5 else "Negativo"
    print(f"\nTexto: {sentence}")
    print(f"Predicción numérica: {pred:.4f}")
    print(f"Sentimiento predicho: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.0000e+00 - loss: 0.7010
Accuracy: 0.00%
Oraciones originales( 5 ):
['No fui al estreno de la película porque nadie me quería acompañar', 'Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio', 'Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor', 'Al final decidí no ir al cine porque estaba cansada', 'Todo es maravilloso y formidable, muy bonito']
Oraciones procesadas( 5 ):
[[14  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  7  0  0  0  0  0]
 [ 0  0  0  0  0  5  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0 12  0 25  0  8  0 26  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Predicciones detalladas:

Texto: No fui al estreno de la película porque nadie me quería acompañar
Predicción numérica: 0.5040
Sentimiento predicho: Positivo

Texto: Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio
Predicci